In [24]:
from typing import Annotated, Sequence, TypedDict
from langgraph.graph import StateGraph,START,END
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools import tool
from dotenv import load_dotenv
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain_core.messages import SystemMessage,AIMessage,HumanMessage,ToolMessage,BaseMessage
import os

load_dotenv()

True

In [13]:
llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [16]:
@tool
def addition(a:int,b:int)->int:
    """This function adds two number"""
    return a+b

@tool
def substraction(a:int,b:int)->int:
    """This function substracts two number"""
    return a-b
@tool
def multiplication(a:int,b:int)->int:
    """This function multiplys two number"""
    return a*b

tools=[addition,substraction,multiplication]

In [17]:
tool_binded_llm=llm.bind_tools(tools)

In [20]:
class AgentState(TypedDict):
    messages:Annotated[Sequence[BaseMessage],add_messages]

In [22]:
def llm_call(state:AgentState)->AgentState:
    system_prompt=SystemMessage(content="you are a helpfull ai assistant who helps user in your best way possible")
    response = llm.invoke([system_prompt] + state['messages'])
    return {'messages':[response]}

In [23]:
def should_continue(state:AgentState):
    messages = state["messages"]
    last_message = messages[-1]
    if not last_message.tool_calls: 
        return "end"
    else:
        return "continue"


In [26]:
graph = StateGraph(AgentState)
graph.add_node("agent",llm_call)

tool_node=ToolNode(tools=tools)
graph.add_node('tools',tool_node)

graph.set_entry_point('agent')

graph.add_conditional_edges(
    "agent",
    should_continue,
    {
        "continue": "tools",
        "end": END,
    },
)
graph.add_edge('tools','agent')

app=graph.compile()



In [27]:
inputs = {"messages": [("user", "Add 40 + 12 and then multiply the result by 6. Also tell me a joke please.")]}
app.invoke(input)

InvalidUpdateError: Expected dict, got <bound method Kernel.raw_input of <ipykernel.ipkernel.IPythonKernel object at 0x0000012444195F10>>
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/INVALID_GRAPH_NODE_RETURN_VALUE

In [28]:
from typing import Annotated, Sequence, TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools import tool
from dotenv import load_dotenv
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain_core.messages import SystemMessage, BaseMessage
import os

load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

@tool
def addition(a: int, b: int) -> int:
    """This function adds two numbers"""
    return a + b

@tool
def subtraction(a: int, b: int) -> int:
    """This function subtracts two numbers"""
    return a - b

@tool
def multiplication(a: int, b: int) -> int:
    """This function multiplies two numbers"""
    return a * b

tools = [addition, subtraction, multiplication]

# ✅ Bind tools
tool_binded_llm = llm.bind_tools(tools)

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

def llm_call(state: AgentState) -> AgentState:
    system_prompt = SystemMessage(content="You are a helpful AI assistant who helps the user in the best way possible.")
    # ✅ Use tool-binded LLM
    response = tool_binded_llm.invoke([system_prompt] + state["messages"])
    return {"messages": [response]}

def should_continue(state: AgentState):
    messages = state["messages"]
    last_message = messages[-1]
    if not getattr(last_message, "tool_calls", None):
        return "end"
    else:
        return "continue"

graph = StateGraph(AgentState)
graph.add_node("agent", llm_call)

tool_node = ToolNode(tools=tools)
graph.add_node("tools", tool_node)

graph.set_entry_point("agent")

graph.add_conditional_edges(
    "agent",
    should_continue,
    {
        "continue": "tools",
        "end": END,
    },
)

graph.add_edge("tools", "agent")

app = graph.compile()

# ✅ Fix variable name
inputs = {"messages": [("user", "Add 40 + 12 and then multiply the result by 6. Also tell me a joke please.")]}
result = app.invoke(inputs)
print(result)


{'messages': [HumanMessage(content='Add 40 + 12 and then multiply the result by 6. Also tell me a joke please.', additional_kwargs={}, response_metadata={}, id='5f896c5c-2e9d-45ea-9529-ddccb74b6f38'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'addition', 'arguments': '{"b": 12.0, "a": 40.0}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': []}, id='run--9fc3cd5c-c4a5-42b7-91ec-a9065b1073bb-0', tool_calls=[{'name': 'addition', 'args': {'b': 12.0, 'a': 40.0}, 'id': '78ee3872-2f74-4f75-993c-cd208aee80f9', 'type': 'tool_call'}], usage_metadata={'input_tokens': 180, 'output_tokens': 207, 'total_tokens': 387, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 187}}), ToolMessage(content='52', name='addition', id='d0d259f7-7974-43d3-ad0b-4c069d8a04ec', tool_call_id='78ee3872-2f74-4f75-993c-cd208aee80f9'), AIMessage(content='The s